<a href="https://colab.research.google.com/github/Nakib-Nasrullah/Dengue_Research/blob/main/Hybrid_Ensemble.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Imoort libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [2]:
from google.colab import drive
drive.mount('/content/drive')
import warnings
warnings.filterwarnings('ignore')

Mounted at /content/drive


In [3]:
df = pd.read_csv('/content/drive/MyDrive/Datasets/Dengue_final_dataset_updated.csv')

In [4]:
# Clean columns
df.columns = [c.strip() for c in df.columns]

In [10]:
# Detect target column
target_col = None
for col in ['Dengue_case']:
    if col in df.columns:
        target_col = col
        break

In [11]:
# Date handling
if 'Date' in df.columns:
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values('Date')

In [12]:
# Fill missing
df = df.ffill().bfill()

# **Feature Engineering (Lag Features)**

In [13]:
def create_lag(df, col, lags):
    for lag in lags:
        df[f"{col}_Lag_{lag}"] = df[col].shift(lag)
    return df

df = create_lag(df, target_col, [14,21])

if 'Temperature' in df.columns:
    df = create_lag(df, 'Temperature', [14,21,30])

df = df.bfill()


Feature Selection (XGBoost Importance)

In [14]:
from xgboost import XGBRegressor

features = [c for c in df.columns if 'Lag' in c or c in ['Rainfall','Humidity','Temperature']]
X = df[features]
y = df[target_col]

model_fs = XGBRegressor()
model_fs.fit(X, y)

import pandas as pd

importance = pd.DataFrame({
    'Feature': features,
    'Importance': model_fs.feature_importances_
}).sort_values(by='Importance', ascending=False)

print(importance)

# Select top features
selected_features = importance['Feature'].head(6).tolist()
print("Selected:", selected_features)


               Feature  Importance
14        Dengue_Lag_7    0.096950
3      Rainfall_Lag_14    0.091356
5      Rainfall_Lag_30    0.072691
9          Temp_Lag_30    0.068904
4      Rainfall_Lag_21    0.065675
8          Temp_Lag_21    0.062382
1             Rainfall    0.059371
10      Humidity_Lag_7    0.058102
11     Humidity_Lag_14    0.054019
6           Temp_Lag_7    0.053723
13     Humidity_Lag_30    0.051060
16  Dengue_case_Lag_21    0.048677
7          Temp_Lag_14    0.047072
0             Humidity    0.046823
2       Rainfall_Lag_7    0.046743
15  Dengue_case_Lag_14    0.046274
12     Humidity_Lag_21    0.030178
Selected: ['Dengue_Lag_7', 'Rainfall_Lag_14', 'Rainfall_Lag_30', 'Temp_Lag_30', 'Rainfall_Lag_21', 'Temp_Lag_21']


# **XGBoost Model**

In [15]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

X = df[selected_features]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, shuffle=False, test_size=0.2
)

xgb = XGBRegressor(n_estimators=200, learning_rate=0.05)
xgb.fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_test)

rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)

print("XGB RMSE:", rmse_xgb)
print("XGB MAE:", mae_xgb)


XGB RMSE: 32.086393083264284
XGB MAE: 9.121831893920898


# **LSTM Model**

In [16]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
scaled = scaler.fit_transform(df[selected_features + [target_col]])

def create_seq(data, seq_len=14):
    X, y = [], []
    for i in range(len(data)-seq_len):
        X.append(data[i:i+seq_len, :-1])
        y.append(data[i+seq_len, -1])
    return np.array(X), np.array(y)

X_seq, y_seq = create_seq(scaled)


# **Train LSTM**

In [17]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

lstm_model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(X_seq.shape[1], X_seq.shape[2])),
    LSTM(32),
    Dense(1)
])

lstm_model.compile(optimizer='adam', loss='mse')
lstm_model.fit(X_seq, y_seq, epochs=10, batch_size=32)


Epoch 1/10
730/730 ━━━━━━━━━━━━━━━━━━━━ 16s 17ms/step - loss: 0.0019
Epoch 2/10
730/730 ━━━━━━━━━━━━━━━━━━━━ 11s 15ms/step - loss: 0.0018
Epoch 3/10
730/730 ━━━━━━━━━━━━━━━━━━━━ 10s 14ms/step - loss: 0.0017
Epoch 4/10
730/730 ━━━━━━━━━━━━━━━━━━━━ 11s 16ms/step - loss: 0.0017
Epoch 5/10
730/730 ━━━━━━━━━━━━━━━━━━━━ 12s 17ms/step - loss: 0.0017
Epoch 6/10
730/730 ━━━━━━━━━━━━━━━━━━━━ 10s 14ms/step - loss: 0.0017
Epoch 7/10
730/730 ━━━━━━━━━━━━━━━━━━━━ 10s 14ms/step - loss: 0.0017
Epoch 8/10
730/730 ━━━━━━━━━━━━━━━━━━━━ 11s 16ms/step - loss: 0.0017
Epoch 9/10
730/730 ━━━━━━━━━━━━━━━━━━━━ 12s 17ms/step - loss: 0.0017
Epoch 10/10
730/730 ━━━━━━━━━━━━━━━━━━━━ 20s 16ms/step - loss: 0.0017


# **Attention-Based LSTM**

In [18]:
import tensorflow as tf
from tensorflow.keras.layers import Layer

class Attention(Layer):
    def call(self, inputs):
        score = tf.nn.tanh(inputs)
        weights = tf.nn.softmax(score, axis=1)
        context = weights * inputs
        return tf.reduce_sum(context, axis=1)

from tensorflow.keras.layers import Input
from tensorflow.keras.models import Model

inputs = Input(shape=(X_seq.shape[1], X_seq.shape[2]))
lstm_out = LSTM(64, return_sequences=True)(inputs)
attn_out = Attention()(lstm_out)
output = Dense(1)(attn_out)

attn_model = Model(inputs, output)
attn_model.compile(optimizer='adam', loss='mse')

attn_model.fit(X_seq, y_seq, epochs=10, batch_size=32)


Epoch 1/10
730/730 ━━━━━━━━━━━━━━━━━━━━ 17s 18ms/step - loss: 0.0018
Epoch 2/10
730/730 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - loss: 0.0018
Epoch 3/10
730/730 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - loss: 0.0018
Epoch 4/10
730/730 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - loss: 0.0017
Epoch 5/10
730/730 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - loss: 0.0017
Epoch 6/10
730/730 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - loss: 0.0017
Epoch 7/10
730/730 ━━━━━━━━━━━━━━━━━━━━ 12s 11ms/step - loss: 0.0017
Epoch 8/10
730/730 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - loss: 0.0017
Epoch 9/10
730/730 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - loss: 0.0017
Epoch 10/10
730/730 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - loss: 0.0017


# **Evaluate LSTM + Attention**

In [19]:
y_pred_lstm = lstm_model.predict(X_seq)
y_pred_attn = attn_model.predict(X_seq)

rmse_lstm = np.sqrt(mean_squared_error(y_seq, y_pred_lstm))
rmse_attn = np.sqrt(mean_squared_error(y_seq, y_pred_attn))

print("LSTM RMSE:", rmse_lstm)
print("Attention RMSE:", rmse_attn)


730/730 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step
730/730 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step
LSTM RMSE: 0.04112982270852638
Attention RMSE: 0.041287986879264535


# **Hybrid Model (Final)**

In [20]:
# Align lengths
min_len = min(len(y_pred_xgb), len(y_pred_lstm))

alpha = 0.5
beta = 0.3
gamma = 0.2

hybrid_pred = (
    alpha * y_pred_xgb[:min_len] +
    beta * y_pred_lstm.flatten()[:min_len] +
    gamma * y_pred_attn.flatten()[:min_len]
)

rmse_hybrid = np.sqrt(mean_squared_error(y_test[:min_len], hybrid_pred))
mae_hybrid = mean_absolute_error(y_test[:min_len], hybrid_pred)

print("Hybrid RMSE:", rmse_hybrid)
print("Hybrid MAE:", mae_hybrid)


Hybrid RMSE: 32.27228669123479
Hybrid MAE: 8.962692260742188


# **Future Prediction (7 Days)**

In [21]:
last_seq = X_seq[-1].reshape(1, X_seq.shape[1], X_seq.shape[2])

future = []

for _ in range(7):
    pred = attn_model.predict(last_seq)[0][0]
    future.append(pred)

    last_seq = np.roll(last_seq, -1, axis=1)
    last_seq[0, -1, -1] = pred

print("Next 7 Days:", future)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step
Next 7 Days: [np.float32(0.0067425976), np.float32(0.0060104323), np.float32(0.0059311385), np.float32(0.0060056504), np.float32(0.00589681), np.float32(0.0061139097), np.float32(0.0058868364)]
